<a href="https://colab.research.google.com/github/Deboxa/Cats_dogs_ML_GRS/blob/main/ML_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install tensorflowjs

INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of wheel to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting un

In [6]:
import os
import cv2
from tqdm import tqdm

SRC_DIR = '/content/drive/MyDrive/cat_n_dog_dataset'
DST_DIR = '/content/data_processed'
IMG_SIZE = 100

os.makedirs(DST_DIR, exist_ok=True)
for folder in ['cats', 'dogs']:
    os.makedirs(os.path.join(DST_DIR, folder), exist_ok=True)

source_folders = {'Cat': 'cats', 'Dog': 'dogs'}

for src_folder, dst_folder in source_folders.items():
    src_path = os.path.join(SRC_DIR, src_folder)
    dst_path = os.path.join(DST_DIR, dst_folder)
    for filename in tqdm(os.listdir(src_path), desc=src_folder):
        filepath = os.path.join(src_path, filename)
        image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
        if image is None:
            continue
        resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        name, ext = os.path.splitext(filename)
        out_path = os.path.join(dst_path, name + '.jpg')
        cv2.imwrite(out_path, resized, [cv2.IMWRITE_JPEG_QUALITY, 95])

Dog: 100%|██████████| 12499/12499 [03:02<00:00, 68.48it/s] 


In [7]:
import tensorflow as tf  # type: ignore
from tensorflow.keras import layers, models # type: ignore
import tensorflowjs as tfjs # type: ignore

tf.random.set_seed(1)
IMG_SIZE = 100
BATCH_SIZE = 32
EPOCHS = 20
TRAIN_DIR = '/content/data_processed'

train_ds = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, validation_split=0.2, subset="training", seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, color_mode='grayscale', label_mode='binary') # type: ignore
val_ds = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, validation_split=0.2, subset="validation", seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, color_mode='grayscale', label_mode='binary') # type: ignore

normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

data_augmentation = tf.keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.1), layers.RandomZoom(0.1),]) # type: ignore
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True) # type: ignore
checkpoint = tf.keras.callbacks.ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, mode='max') # type: ignore

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stop, checkpoint])

model.save('cats_dogs_final_model.keras')
tfjs.converters.save_keras_model(model, 'tfjs_model')

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Validation accuracy: {val_acc*100:.2f}%")

Found 24997 files belonging to 2 classes.
Using 19998 files for training.
Found 24997 files belonging to 2 classes.
Using 4999 files for validation.
Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 46ms/step - accuracy: 0.5594 - loss: 0.6816 - val_accuracy: 0.6197 - val_loss: 0.6589
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.6571 - loss: 0.6239 - val_accuracy: 0.6827 - val_loss: 0.5841
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.7116 - loss: 0.5661 - val_accuracy: 0.7742 - val_loss: 0.4868
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 41ms/step - accuracy: 0.7491 - loss: 0.5163 - val_accuracy: 0.7868 - val_loss: 0.4563
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 41ms/step - accuracy: 0.7693 - loss: 0.4879 - val_accuracy: 0.8012 - val_loss: 0.4286
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.7818 - loss: 0.4632 - val_accuracy: 0.8156 - val_loss: 0.3968
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.793

failed to lookup keras version from the file,
    this is likely a weight only file
Validation accuracy: 84.10%


#It is better to run it on a T4 GPU; the speed is 4–5 times faster per epoch.